In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_opt_L-ala_cell-17O_magres.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-62.93377238 166.22451312 -80.56157078]
 [172.4697815   65.96113206  24.7350798 ]
 [  8.22591105 -25.69604004 -51.81500928]]

17O2 sigma:
 [[ -53.82014758  218.76780944   55.02898048]
 [ 185.81011053  115.33568175  -88.18842382]
 [  18.64191826  -51.17834508 -172.26916506]]

17O3 sigma:
 [[-62.93377238 166.22451312  80.56157078]
 [172.4697815   65.96113206 -24.7350798 ]
 [ -8.22591105  25.69604004 -51.81500928]]

17O4 sigma:
 [[ -62.93377238 -166.22451312   80.56157078]
 [-172.4697815    65.96113206   24.7350798 ]
 [  -8.22591105  -25.69604004  -51.81500928]]

17O5 sigma:
 [[ -62.93377238 -166.22451312  -80.56157078]
 [-172.4697815    65.96113206  -24.7350798 ]
 [   8.22591105   25.69604004  -51.81500928]]

17O6 sigma:
 [[ -53.82014758  218.76780944  -55.02898048]
 [ 185.81011053  115.33568175   88.18842382]
 [ -18.64191826   51.17834508 -172.26916506]]

17O7 sigma:
 [[ -53.82014758 -218.76780944  -55.02898048]
 [-185.81011053  115.33568175  -88.18842382]
 [ -18.64191826

In [6]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.3985376856896785

17O2 sigma:
 8.257931708019381

17O3 sigma:
 6.398537685689589

17O4 sigma:
 6.398537685689671

17O5 sigma:
 6.398537685689582

17O6 sigma:
 8.25793170801937

17O7 sigma:
 8.257931708019346

17O8 sigma:
 8.257931708019344



In [7]:
Q = -0.0256 #electric quadrupole moment for O17 in barn

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = -53.8201; CS_total[0,1] = 218.7678; CS_total[0,2] = 55.0290;
CS_total[1,0] = 185.8101; CS_total[1,1] = 115.3357; CS_total[1,2] = -88.1884;
CS_total[2,0] = 18.6419; CS_total[2,1] = -51.1783; CS_total[2,2] =  -172.2692;

Cs = np.zeros((3,3)) # CS symmetric (l = 0 + l = 2) 

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= 0.7069; efg[0,1]= -0.1586; efg[0,2]=  0.1535;
efg[1,0]= efg[0,1]; efg[1,1]= 0.6211 ; efg[1,2]= -0.2739;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= -1.3279;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 

V = efg*Q*234.9647

print(V)

print(Cs)

[[-4.25207159  0.95399428 -0.92331729]
 [ 0.95399428 -3.73597632  1.64753488]
 [-0.92331729  1.64753488  7.9874464 ]]
[[ -53.8201   202.28895   36.83545]
 [ 202.28895  115.3357   -69.68335]
 [  36.83545  -69.68335 -172.2692 ]]


In [8]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [-5.21465155 -3.05005746  8.2641075 ] 

 Unsorted Eigenvectors:
 [[-0.75898827  0.64804767 -0.06301608]
 [ 0.63745253  0.75930109  0.13082862]
 [-0.13263136 -0.05912763  0.98940024]] 

Sorted Eigenvalues: 
 [-3.05005746 -5.21465155  8.2641075 ] 

Sorted Eigenvectors: 
 [[ 0.64804767 -0.75898827 -0.06301608]
 [ 0.75930109  0.63745253  0.13082862]
 [-0.05912763 -0.13263136  0.98940024]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 253.42238608 -112.77889617 -251.39708991] 

 Unsorted Eigenvectors:
 [[-0.54009906 -0.6024901  -0.58762121]
 [-0.83674993  0.3095575   0.45168983]
 [ 0.0902361  -0.73564926  0.67132527]] 

Sorted Eigenvalues: 
 [-112.77889617 -251.39708991  253.42238608] 

Sorted Eigenvectors: 
 [[-0.6024901  -0.58762121 -0.54009906]
 [ 0.3095575   0.45168983 -0.83674993]
 [-0.73564926  0.67132527  0.0902361 ]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.050057462243091 -5.214651548625457 8.264107501236547
CSA Tensor Components δyy, δxx, δzz: 
 -112.77889617007393 -251.39708990615705 253.42238607623102


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        8.26411
etaq            0.261927
iso_cs (ppm)  -36.9179
csa (ppm)     290.34
etas            0.477434


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.75898827  0.64804767 -0.06301608]
 [ 0.63745253  0.75930109  0.13082862]
 [-0.13263136 -0.05912763  0.98940024]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
24.027516371234306 8.349681499095873 64.2813275652361 

Direction cosine csa: 

[[-0.58762121 -0.6024901  -0.54009906]
 [ 0.45168983  0.3095575  -0.83674993]
 [ 0.67132527 -0.73564926  0.0902361 ]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-47.617622801140655 84.82281038068325 -57.15887004753798 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -40.50001469509406 chi: 89.20678898656453 xi: -82.21590002857248 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[ -53.8201   202.28895   36.83545]
 [ 202.28895  115.3357   -69.68335]
 [  36.83545  -69.68335 -172.2692 ]]
CSA Tensor in Tenon Frame: 
 [[ 173.11354143 -135.36096404 -102.62603309]
 [-135.36096404 -207.65794143   42.04244559]
 [-102.62603309   42.04244559  -76.2092    ]]
Quad Tensor in Crystal Frame: 
 [[-4.25207159  0.95399428 -0.92331729]
 [ 0.95399428 -3.73597632  1.64753488]
 [-0.92331729  1.64753488  7.9874464 ]]
Quad Tensor in Tenon Frame: 
 [[-3.70689293 -0.6553474  -0.74892688]
 [-0.6553474   2.7619213  -6.34622216]
 [-0.74892688 -6.34622216  0.94437012]]
